# 04 · The 8-bit corpus latent table

On the board the search (similarities between the query and every grain, then the penalties and the softmax) is the
only analysis stage whose cost grows with the corpus (notebook 05). Quantising the matching table to 8 bits cuts its
memory traffic by almost 4× and lets the similarities run as integer dot products.

The scheme the plugin uses (`row`, `distillation/int8_table.py`, `plugin/src/Matcher.cpp` with
`ELKGRANULAR_TABLE=int8`): one scale per grain, s_i = max|row_i| / 127. Each query frame is centred, normalised and
quantised with its own scale s_q, and the similarity is s_i · s_q · (integer dot product). A second scheme (`dim`, one
scale per latent dimension) is reported for comparison. The question is whether quantisation changes the ordering
that selection depends on.

In [1]:
import json

import numpy as np
import torch

from analysis import demo, report
from analysis.board_logs import load_runs
from corpus_builder import elkgrain
from distillation import student as S
from distillation.evaluate import spearman_rows
from distillation.int8_table import centred_queries, quantise_table, sims_f32, sims_int8

pk = elkgrain.ElkGrainPack(demo.ensure_pack())
book, centre = np.asarray(pk.emb, np.float32), np.asarray(pk.centre, np.float32)
queries = demo.encode_queries()
mlp, mu, sd = S.mlp_from_json()
print(pk)

ElkGrainPack(demo.elkgrain: 633 grains, unit=1, hop48=4486, margin48=480, 3 files, 59.3s audio, 6.0 MB)


## 1. Quantise the demo table

In [2]:
tabs = {s: quantise_table(book, s) for s in ("row", "dim")}
print(f"32-bit table: {book.nbytes:,} B ({book.shape[1]} × 4 B per grain)")
for s, (bq, sc, nbytes) in tabs.items():
    print(f"8-bit '{s}':   {nbytes:,} B (int8 rows + {sc.size} float32 scales), {book.nbytes / nbytes:.2f}× smaller")
bq, sc, _ = tabs["row"]
print("largest reconstruction error of a table entry:", float(np.abs(bq * sc[:, None] - book).max()))

32-bit table: 162,048 B (64 × 4 B per grain)
8-bit 'row':   43,044 B (int8 rows + 633 float32 scales), 3.76× smaller
8-bit 'dim':   40,768 B (int8 rows + 64 float32 scales), 3.97× smaller
largest reconstruction error of a table entry: 0.0018826574087142944


## 2. Does the ordering change?

For each held-out demo query, the teacher latents and the student embeddings are compared against the 32-bit and
the 8-bit table: Spearman ρ between the two orderings of all grains, agreement of the best grain, and the largest
change of a similarity.

In [3]:
rows = []
for qid, q in queries.items():
    with torch.no_grad():
        b = mlp(torch.tensor((q["mel80"] - mu) / sd)).numpy().T
    for cond, target in (("teacher", q["lat"]), ("student", b)):
        Q = centred_queries(centre, target)
        Sf = sims_f32(book, Q)
        for s, (bq, sc, _) in tabs.items():
            Si = sims_int8(bq, sc, s, Q)
            rows.append({"q": qid, "cond": cond, "scheme": s, "rho": float(np.mean(spearman_rows(Si, Sf))),
                         "top1": float(np.mean(Si.argmax(1) == Sf.argmax(1))), "err": float(np.abs(Si - Sf).max())})
report.table(rows, [("q", "Query"), ("cond", "Query encoder"), ("scheme", "8-bit scheme"),
                    ("rho", "ρ, 8-bit vs 32-bit ordering", ".5f"), ("top1", "same best grain", ".3f"), ("err", "largest change of a similarity", ".4f")])

| Query | Query encoder | 8-bit scheme | ρ, 8-bit vs 32-bit ordering | same best grain | largest change of a similarity |
|---|---|---|---|---|---|
| qp1 | teacher | row | 0.99998 | 0.988 | 0.0044 |
| qp1 | teacher | dim | 0.99997 | 0.988 | 0.0050 |
| qp1 | student | row | 0.99998 | 0.941 | 0.0043 |
| qp1 | student | dim | 0.99997 | 0.918 | 0.0055 |
| qt1 | teacher | row | 0.99997 | 0.988 | 0.0053 |
| qt1 | teacher | dim | 0.99997 | 0.988 | 0.0058 |
| qt1 | student | row | 0.99998 | 0.976 | 0.0043 |
| qt1 | student | dim | 0.99997 | 0.941 | 0.0058 |
| qx1 | teacher | row | 0.99998 | 0.988 | 0.0045 |
| qx1 | teacher | dim | 0.99997 | 0.976 | 0.0056 |
| qx1 | student | row | 0.99997 | 0.953 | 0.0045 |
| qx1 | student | dim | 0.99997 | 0.953 | 0.0050 |

The orderings are the same to within a few swaps of near-equal grains: ρ is 1.000 at three decimals. The best grain
changes on a few per cent of frames, only where two grains are within the quantisation step of each other, which the
τ = 0.05 sampling does not distinguish anyway.

## 3. The paper's corpora

`results/exp1/int8_vs_f32.json` is the same study (`python -m distillation.int8_table eval`) over the paper's four
corpora and 12 held-out queries (τ = 0.05, seed 0), regenerated on 23 Sep 2026 bit-identical to the 16 Sep run.

In [4]:
d = json.loads((demo.REPO / "results/exp1/int8_vs_f32.json").read_text())
R = d["rows"]
def m(key, **sel):
    xs = [r[key] for r in R if all(r[k] == v for k, v in sel.items())]
    return float(np.mean(xs)), float(np.min(xs))
out = []
for c, t in d["tables"].items():
    for cond in ("O", "B"):
        rho, rho_min = m("rho_vs_f32", pack=c, cond=cond, scheme="row")
        out.append({"corpus": c, "grains": t["grains"], "cond": {"O": "teacher", "B": "student"}[cond],
                    "f32": t["bytes_f32"], "row": t["bytes_row"], "ratio": t["bytes_f32"] / t["bytes_row"],
                    "rho": rho, "rho_min": rho_min, "top1": m("top1_vs_f32", pack=c, cond=cond, scheme="row")[0],
                    "pct": m("pct_in_f32", pack=c, cond=cond, scheme="row")[0],
                    "pct_O": m("orc_pct_live", pack=c, cond=cond, scheme="row")[0]})
report.table(out, [("corpus", "Corpus"), ("grains", "Grains", ","), ("cond", "Query encoder"), ("f32", "32-bit (B)", ","),
                   ("row", "8-bit (B)", ","), ("ratio", "×", ".2f"), ("rho", "ρ 8 vs 32-bit", ".4f"),
                   ("rho_min", "lowest ρ of a query", ".5f"), ("top1", "same best grain", ".3f"),
                   ("pct", "Pct. of the 8-bit best grain in the 32-bit ordering", ".4f"),
                   ("pct_O", "Pct. vs teacher, live selection", ".3f")])
student_rho = min(r["rho_vs_f32"] for r in R if r["cond"] == "B" and r["scheme"] == "row")
board = {(r.grains, r.table): r.header["table_bytes"] for r in load_runs()}
print(f"lowest ρ between the student's 32- and 8-bit orderings over all corpora and queries: {student_rho:.5f}")
print("table bytes in the board's worker logs:", {k: f"{v:,}" for k, v in sorted(board.items())})

| Corpus | Grains | Query encoder | 32-bit (B) | 8-bit (B) | × | ρ 8 vs 32-bit | lowest ρ of a query | same best grain | Pct. of the 8-bit best grain in the 32-bit ordering | Pct. vs teacher, live selection |
|---|---|---|---|---|---|---|---|---|---|---|
| perc | 5,115 | teacher | 1,309,440 | 347,820 | 3.76 | 1.0000 | 0.99998 | 0.969 | 1.0000 | 0.975 |
| perc | 5,115 | student | 1,309,440 | 347,820 | 3.76 | 1.0000 | 0.99998 | 0.967 | 1.0000 | 0.935 |
| tonal | 4,900 | teacher | 1,254,400 | 333,200 | 3.76 | 1.0000 | 0.99997 | 0.982 | 1.0000 | 0.983 |
| tonal | 4,900 | student | 1,254,400 | 333,200 | 3.76 | 1.0000 | 0.99998 | 0.985 | 1.0000 | 0.956 |
| text | 4,554 | teacher | 1,165,824 | 309,672 | 3.76 | 1.0000 | 0.99997 | 0.979 | 1.0000 | 0.976 |
| text | 4,554 | student | 1,165,824 | 309,672 | 3.76 | 1.0000 | 0.99997 | 0.983 | 1.0000 | 0.959 |
| field | 12,018 | teacher | 3,076,608 | 817,224 | 3.76 | 1.0000 | 0.99996 | 0.984 | 1.0000 | 0.978 |
| field | 12,018 | student | 3,076,608 | 817,224 | 3.76 | 1.0000 | 0.99997 | 0.981 | 1.0000 | 0.933 |

lowest ρ between the student's 32- and 8-bit orderings over all corpora and queries: 0.99997
table bytes in the board's worker logs: {(1107, 'f32'): '283,392', (1107, 'int8'): '75,276', (5115, 'f32'): '1,309,440', (5115, 'int8'): '347,820', (12018, 'f32'): '3,076,608', (12018, 'int8'): '817,224'}


The board's worker logs report the same table sizes as this study (`table_bytes`, 3,076,608 B in 32 bits and
817,224 B in 8 bits for the field corpus), so the board ran exactly this quantisation. How much faster the 8-bit
search is on the board is in notebook 05. (`plugin/tools/int8_parity.py` checks that the C++ int8 table is
byte-identical to the NumPy one and that both select the same grains.)

## Numbers of the paper this notebook reproduces

In [5]:
f = d["tables"]["field"]
ok = report.checks([
    ("Results: ρ between the student's 32- and 8-bit orderings", "ρ = 1.000", f"{student_rho:.3f} (lowest query {student_rho:.5f})",
     f"{student_rho:.3f}" == "1.000"),
    ("Section 4: the latent table quantised to 8 bits (field corpus, board log)", "8-bit",
     f"{board[(12018, 'int8')]:,} B = int8 rows + scales ({f['bytes_row']:,} B here)", board[(12018, "int8")] == f["bytes_row"]),
])
print("all equal:", ok)

| Paper element | Printed in the paper | Computed here | Equal |
|---|---|---|---|
| Results: ρ between the student's 32- and 8-bit orderings | ρ = 1.000 | 1.000 (lowest query 0.99997) | ✓ |
| Section 4: the latent table quantised to 8 bits (field corpus, board log) | 8-bit | 817,224 B = int8 rows + scales (817,224 B here) | ✓ |

all equal: True
